In [0]:
%run ../source_to_bronze/utils

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [0]:
# Custom schema for employee
employee_schema = StructType([
    StructField("EmployeeID", IntegerType(), True),
    StructField("EmployeeName", StringType(), True),
    StructField("Department", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("Salary", IntegerType(), True),
    StructField("Age", IntegerType(), True)
])

# Custom schema for department
department_schema = StructType([
    StructField("DepartmentID", StringType(), True),
    StructField("DepartmentName", StringType(), True)
])

# Custom schema for country
country_schema = StructType([
    StructField("CountryCode", StringType(), True),
    StructField("CountryName", StringType(), True)
])

In [0]:
# Method 1
employee_df = (
    spark.read
         .format("csv")
         .option("header", "true")
         .schema(employee_schema)
         .load("/Volumes/workspace/default/source_to_bronze/employee.csv")
)

# Method 2
department_df = (
    spark.read
         .option("header", "true")
         .schema(department_schema)
         .csv("/Volumes/workspace/default/source_to_bronze/department_df.csv")
)

# Method 3
country_df = spark.read.csv(
    "/Volumes/workspace/default/source_to_bronze/country_df.csv",
    header=True,
    schema=country_schema
)

In [0]:
display(employee_df)
display(department_df)
display(country_df)

In [0]:
employee_df = rename_columns_to_snake(employee_df)
department_df = rename_columns_to_snake(department_df)
country_df = rename_columns_to_snake(country_df)

In [0]:
from pyspark.sql.functions import current_date

In [0]:
# Add load_date
employee_df = employee_df.withColumn("load_date", current_date())
department_df = department_df.withColumn("load_date", current_date())
country_df = country_df.withColumn("load_date", current_date())

In [0]:
display(employee_df)
display(department_df)
display(country_df)

In [0]:
from pyspark.sql.functions import col

dim_employee_df = (
    employee_df.alias("e")
    .join(
        department_df.alias("d"),
        col("e.department") == col("d.department_id"),
        "left"
    )
    .join(
        country_df.alias("c"),
        col("e.country") == col("c.country_code"),
        "left"
    )
    .select(
        col("e.employee_id"),
        col("e.employee_name"),
        col("e.department").alias("department_id"),
        col("d.department_name"),
        col("e.country").alias("country_code"),
        col("c.country_name"),
        col("e.salary"),
        col("e.age"),
        col("e.load_date")
    )
)

In [0]:
display(dim_employee_df)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS employee_info")

In [0]:
silver_path = "/Volumes/workspace/default/silver/employee_info/dim_employee"

In [0]:
(
    dim_employee_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("employee_info.dim_employee")
)

In [0]:
# Verify silver table
display(spark.table("employee_info.dim_employee"))